# Notatnik do stworzenia pierwszych modeli prognozowania #

## 1. Załadowanie bibliotek ##

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import sys 
import os
from statsmodels.api import OLS

sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data

## 2. Załadowanie danych i stworzenie zbiorow treningowych i testowych ##


In [3]:
# załadowanie danych
df_ready = load_data('../data/processed/sp500_ready.csv')

✅ Ustawiono indeks czasowy na kolumnie: Date


In [4]:
# Podział na X i y
features = ['mom_RSI', 'mr_DistSMA200', 'vol_RollingStd', 'lag_return']
X = df_ready[features]
X['const'] = 1
y = df_ready['target_return']

print(X.head())
print(y.head())

              mom_RSI  mr_DistSMA200  vol_RollingStd  lag_return  const
Date                                                                   
2023-10-19  48.770040       0.011043        0.132895   -0.016601      1
2023-10-20  43.049454      -0.002119        0.143047   -0.024234      1
2023-10-23  48.487434      -0.004282        0.130799   -0.036460      1
2023-10-24  48.023711       0.002535        0.129815   -0.029122      1
2023-10-25  42.269224      -0.012184        0.143131   -0.030075      1
Date
2023-10-19   -0.033459
2023-10-20   -0.025606
2023-10-23   -0.011980
2023-10-24   -0.012766
2023-10-25    0.012129
Name: target_return, dtype: float64


In [5]:
# Podział na zbiór treningowy i testowy
split = int(len(df_ready) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

## 3. Stworzenie modelu zerowego - prognozowanie średniej ##


In [6]:
# Prognoza dla całego zbioru testoweg - srednia ze zbioru treningowego
y_zero_pred = np.full(len(y_test), y_train.mean())


## 4. Stworzenie modelu regresji liniowej ##

In [7]:
# Estymacja modelu 
model = LinearRegression()
model.fit(X_train, y_train)

OLS(y_train, X_train).fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          target_return   R-squared:                       0.039
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                     5.004
Date:                Sat, 02 May 2026   Prob (F-statistic):           0.000582
Time:                        19:53:29   Log-Likelihood:                 1241.0
No. Observations:                 502   AIC:                            -2472.
Df Residuals:                     497   BIC:                            -2451.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
mom_RSI        -1.963e-05   7.62e-05     -0.258      0.797      -0.000       0.000
mr_DistSMA200     -0.0449      0.029     -1.558      0.120      -0.102       0.012
vol_RollingStd     0.0150      0.017      0.903      0.367      -0.018       0.048
lag_return        -0.0798      0.054     -1.481      0.139      -0.186       0.026
const              0.0075      0.006      1.172      0.242      -0.005       0.020
==============================================================================
Omnibus:                      194.332   Durbin-Watson:                   0.432
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1181.994
Skew:                          -1.567   Prob(JB):                    2.15e-257
Kurtosis:                       9.833   Cond. No.                     3.74e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.74e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Predykcja na zbiorze testowym ###

In [8]:
y_pred = model.predict(X_test)

## 5. Porównanie wyników modeli ##

In [9]:
mse_zero = mean_squared_error(y_test, y_zero_pred)
mse_model = mean_squared_error(y_test, y_pred)

In [10]:
# --- PORÓWNANIE ---
print(f"MSE Model Zerowy:   {mse_zero:.6f}")
print(f"MSE regresja liniowa: {mse_model:.6f}")
print(f"R^2 Score:          {r2_score(y_test, y_pred):.4f}")

# Procent trafionych kierunków (Directional Accuracy)
hits_zero = np.sign(y_zero_pred) == np.sign(y_test)
acc_zero = hits_zero.mean()

hits = np.sign(y_pred) == np.sign(y_test)
acc = hits.mean()

print(f"Trafność kierunku (Directional Accuracy) dla modelu zerowego: {acc_zero:.2%}")
print(f"Trafność kierunku (Directional Accuracy) dla regresji liniowej: {acc:.2%}")

MSE Model Zerowy:   0.000346
MSE regresja liniowa: 0.000334
R^2 Score:          0.0175
Trafność kierunku (Directional Accuracy) dla modelu zerowego: 54.76%
Trafność kierunku (Directional Accuracy) dla regresji liniowej: 54.76%


In [ ]:
# weryfikacja prognoz dla regresji liniowej - na razie pokazuje same plusy
print(pd.Series(np.sign(y_pred)).value_counts())

1.0    126
Name: count, dtype: int64
